# SymboLM Robust 6-Arm 4-Budget Evaluation Framework

**Principal Systems Architect & Research Director:** **Sanish Gyawali** (Rupandehi, Nepal)  
**AI Systems Collaborator:** Antigravity (Google DeepMind)  
**Adversarial Scientific Protocol:** Claude (Haiku 5.5, Anthropic)  
**Primary Model:** `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` + `gyawalisanish0/SymboLM-2.1-DeepSeek-1.5B`  
**Control Model (Arm F):** Vanilla `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` (Zero Adapter)  
**Hardware Target:** Kaggle Cloud NVIDIA Tesla T4 GPU  
**Pre-Registration Reference:** `PREREG_VALUATION.md` (Multi-Budget Pareto Frontier Protocol)  

### Core Framework Specifications:
1. **6 Evaluation Arms:**
   - Arm A: Ordinary CoT (Greedy English reasoning)
   - Arm B: Answer-Only (Direct answer baseline)
   - Arm C: Full Registers (SymboLM Plan + Deduce + Verify)
   - Arm D: Router-Only Control (Bypass on facts, CoT on math)
   - Arm E: CoT + Python Calculator Sandbox
   - **Arm F: Vanilla DeepSeek Base (No Adapter — External Baseline)**
2. **Independent Multi-Budget Grid:** $\{256, 512, 1024, 2048\}$ tokens per arm.
3. **Dual-Layer Answer Verification:** Strict string/float comparison + SymPy symbolic algebraic equality.
4. **Effect Size Analysis:** Cohen's d & Common Language Effect Size (CLES) on $J/\text{correct}$ distributions.
5. **Stratified Wilson 95% CIs:** Per-category binomial confidence intervals.
6. **Cryptographic Manifest:** Run manifest with dataset SHA-256, git commit hash, seeds, and hardware specs.

In [ ]:
# Cell 1: Environment & GPU Hardware Verification
import os
import sys
import time
import torch

print('Python Version:', sys.version)
print('PyTorch Version:', torch.__version__)
if torch.cuda.is_available():
    print(f'[OK] CUDA Detected! Count: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name} ({props.total_memory / 1e9:.2f} GB VRAM)')
    !nvidia-smi
else:
    print('[WARNING] CUDA not detected! Running in CPU fallback mode.')


In [ ]:
# Cell 2: Dependencies Installation
!pip install -q --upgrade pip
!pip uninstall -y -q torchao
!pip install -q 'transformers>=4.48.0' 'peft>=0.14.0' 'accelerate>=1.2.0' 'sympy>=1.12' 'datasets>=3.2.0'


In [ ]:
# Cell 3: Clone Production Repository from GitHub
!rm -rf /kaggle/working/symboLM
!git clone -b main https://github.com/gyawalisanish0/symboLM.git /kaggle/working/symboLM
%cd /kaggle/working/symboLM
!git log -n 3 --oneline


In [ ]:
# Cell 3.5: Multi-Session Quota Continuation & Checkpoint Auto-Ingestion
import os
import shutil
from pathlib import Path

robust_dir = Path('/kaggle/working/symboLM/results/robust_eval')
robust_dir.mkdir(parents=True, exist_ok=True)

restored = 0
for tar_arc in Path('/kaggle/input').glob('**/*symbolm*robust*eval*receipts*.tar.gz'):
    print(f'[RESUME ENGINE] Found prior session tarball at: {tar_arc}. Unpacking...')
    os.system(f'tar -xzf "{tar_arc}" -C /kaggle/working/symboLM/')
    restored += 1
    break

for cand in Path('/kaggle/input').glob('**/robust_eval_results*.jsonl'):
    dest = robust_dir / 'robust_eval_results.jsonl'
    if not dest.exists():
        print(f'[RESUME ENGINE] Restoring robust eval checkpoint: {cand} -> {dest}')
        shutil.copy2(cand, dest)
        restored += 1

if restored > 0:
    print('[OK] Checkpoint continuation active! Prior progress loaded successfully.')
else:
    print('[INFO] Clean initial session run. No prior checkpoints detected.')


In [ ]:
# Cell 4: Fuse Flagship DeepSeek-R1-Distill-Qwen-1.5B with SymboLM v2.1 Adapter
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

fused_dir = '/kaggle/working/symbolm_deepseek_fused'
if not os.path.exists(fused_dir):
    print('Fusing DeepSeek-R1-Distill-Qwen-1.5B and SymboLM v2.1 adapter...')
    base_model_id = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
    adapter_id = 'gyawalisanish0/SymboLM-2.1-DeepSeek-1.5B'
    
    tokenizer = AutoTokenizer.from_pretrained(adapter_id, trust_remote_code=True)
    base = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        torch_dtype=torch.float16,
        device_map='auto',
        trust_remote_code=True,
    )
    base.resize_token_embeddings(151712)
    model = PeftModel.from_pretrained(base, adapter_id)
    merged_model = model.merge_and_unload()
    
    merged_model.save_pretrained(fused_dir)
    tokenizer.save_pretrained(fused_dir)
    print(f'[OK] Standalone fused DeepSeek weights saved to: {fused_dir}')
else:
    print(f'[OK] Fused weights already exist at: {fused_dir}')


In [ ]:
# Cell 5: Execute Hardened Arm C Rerun on Tesla T4 (Preserving Arms A, B, D, E, F)
%cd /kaggle/working/symboLM
!python eval/robust_eval.py \
    --input data/held_out_eval_200_clean.jsonl \
    --adapter /kaggle/working/symbolm_deepseek_fused \
    --base-model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
    --out-dir results/robust_eval \
    --report ROBUST_EVAL_REPORT.md \
    --arms C \
    --replace-arms C \
    --budgets 256 512 1024 2048 \
    --device cuda \
    --backup-every 10


In [ ]:
# Cell 6: Display Master Robust Evaluation Report and Package Receipts Archive
import os
from pathlib import Path

print('=' * 80)
print('ROBUST 6-ARM 4-BUDGET EVALUATION REPORT:')
print('=' * 80)
rep = Path('/kaggle/working/symboLM/ROBUST_EVAL_REPORT.md')
if rep.exists():
    print(rep.read_text(encoding='utf-8'))

# Package receipts
!tar -czvf /kaggle/working/symbolm_robust_eval_receipts.tar.gz \
    -C /kaggle/working/symboLM \
    ROBUST_EVAL_REPORT.md \
    results/robust_eval/

print('[DONE] Robust evaluation receipts archived to /kaggle/working/symbolm_robust_eval_receipts.tar.gz')
